<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/Copy_of_RL_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 1: Remove unnecessary columns

1. Pure Identifiers (No Predictive Value)
Remove:
*   Customer Id
*   Category Id
*   Department Id


2. Personally Identifiable / Customer Detail Columns
Remove:
* Customer Email
* Customer Password
* Customer Fname
* Customer Lname
* Customer Street
* Customer Zipcode

3. High-Cardinality Text Fields
* Product Description
* Product Image
* Product Name
* Customer Full Name

5. Duplicate or Redundant Geographic Columns
* Order Region
* Customer Segment
* Order City
* Order State



In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

In [21]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Product Card Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',
]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns
]

# Drop columns
df_rl = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df_rl.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df_rl.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(180519, 37)


In [22]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df_rl.to_csv(output_file, index=False)

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


Identify missing or erroneous data


In [23]:
df = pd.read_csv('dataco_rl_cleaned.csv')

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(180519, 37)
Order Zipcode    155679
dtype: int64


In [24]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [25]:
numeric_cols = df.select_dtypes(include=np.number).columns

# Display summary statistics
print(df[numeric_cols].describe())

       Days for shipping (real)  Days for shipment (scheduled)  \
count             180519.000000                  180519.000000   
mean                   3.497654                       2.931847   
std                    1.623722                       1.374449   
min                    0.000000                       0.000000   
25%                    2.000000                       2.000000   
50%                    3.000000                       4.000000   
75%                    5.000000                       4.000000   
max                    6.000000                       4.000000   

       Benefit per order  Sales per customer  Late_delivery_risk  \
count      180519.000000       180519.000000       180519.000000   
mean           21.974989          183.107609            0.548291   
std           104.433526          120.043670            0.497664   
min         -4274.979980            7.490000            0.000000   
25%             7.000000          104.379997            0.000000 

In [26]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)'
]

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [27]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        ((df[col] < lower_bound) |
         (df[col] > upper_bound))
    ).sum()

    print(f"{col}: {outlier_count} outliers")

Days for shipping (real): 0 outliers
Days for shipment (scheduled): 0 outliers
Benefit per order: 18942 outliers
Sales per customer: 1943 outliers
Late_delivery_risk: 0 outliers
Latitude: 9 outliers
Longitude: 1414 outliers
Order Customer Id: 1198 outliers
Order Id: 0 outliers
Order Item Cardprod Id: 0 outliers
Order Item Discount: 7537 outliers
Order Item Discount Rate: 0 outliers
Order Item Id: 0 outliers
Order Item Product Price: 2048 outliers
Order Item Profit Ratio: 17300 outliers
Order Item Quantity: 0 outliers
Sales: 488 outliers
Order Item Total: 1943 outliers
Order Profit Per Order: 18942 outliers
Order Zipcode: 0 outliers
Product Category Id: 0 outliers
Product Price: 2048 outliers
Product Status: 0 outliers


In [28]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
155679

Final Dataset Shape:
(180519, 37)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                   0  Sportin

# STEP 2: Feature engineering

In [29]:
import pandas as pd
import numpy as np


df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 37)


In [30]:
# DATE FEATURE ENGINEERING


# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)




In [31]:
# PROFIT ENGINEERING


# Create profit margin percentage
df['profit_margin'] = (
    df['Order Profit Per Order'] /
    (df['Sales'] + 1e-6)
)

# Revenue per item
df['revenue_per_item'] = (
    df['Sales'] /
    (df['Order Item Quantity'] + 1e-6)
)


# SHIPPING ENGINEERING


# Shipping urgency score
# Lower scheduled days = more urgent
df['shipping_urgency'] = (
    1 / (df['Days for shipment (scheduled)'] + 1)
)

# Encode shipping speed categories
shipping_map = {
    'Same Day': 4,
    'First Class': 3,
    'Second Class': 2,
    'Standard Class': 1
}

df['shipping_mode_encoded'] = (
    df['Shipping Mode']
    .map(shipping_map)
    .fillna(0)
)




In [32]:
# ROUTE ENGINEERING
df['shipping_route'] = (
    df['Order Region'].astype(str) + '_' +
    df['Customer Country'].astype(str) + '_' +
    df['Shipping Mode'].astype(str)
)

In [33]:
# PRODUCT / ORDER ENGINEERING


# Total items ordered
df['total_order_value_per_quantity'] = (
    df['Order Item Total'] /
    (df['Order Item Quantity'] + 1e-6)
)

# Large order flag
df['large_order'] = (
    df['Order Item Quantity'] >
    df['Order Item Quantity'].median()
).astype(int)

# High profit order flag
df['high_profit_order'] = (
    df['Order Profit Per Order'] >
    df['Order Profit Per Order'].median()
).astype(int)

In [34]:
# GEOGRAPHIC FEATURES


# Combine market and region
if 'Market' in df.columns and 'Order Region' in df.columns:
    df['market_region'] = (
        df['Market'].astype(str) + '_' +
        df['Order Region'].astype(str)
    )

In [35]:
# RL REWARD FEATURE ENGINEERING


# Create normalized profit score
profit_min = df['Order Profit Per Order'].min()
profit_max = df['Order Profit Per Order'].max()

df['normalized_profit'] = (
    (df['Order Profit Per Order'] - profit_min) /
    (profit_max - profit_min + 1e-6)
)

# Simulated lateness proxy
# (Since actual lateness removed to avoid leakage)
# Higher scheduled days may indicate harder shipments

df['lateness_risk_proxy'] = (
    df['Days for shipment (scheduled)'] /
    df['Days for shipment (scheduled)'].max()
)


# ENCODE CATEGORICAL VARIABLES


categorical_cols = [
    'Shipping Mode',
    'Market',
    'Order Region',
    'Category Name',
    'Order Country'
]

# Label encode categoricals
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category').cat.codes

# Encode combined feature if created
if 'market_region' in df.columns:
    df['market_region'] = (
        df['market_region']
        .astype('category')
        .cat.codes
    )




In [36]:
# HANDLE MISSING VALUES


# Fill numeric missing values
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


# REMOVE UNUSED DATE COLUMN


df = df.drop(columns=['order date (DateOrders)'])


# FINAL DATASET CHECK


print("\nFeature Engineering Complete")
print("Final Shape:", df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nSample Features:")
print(df.head())


# SAVE FEATURE-ENGINEERED DATASET


output_file = 'dataco_rl_feature_engineered.csv'

df.to_csv(output_file, index=False)

print(f"\nFeature-engineered dataset saved as: {output_file}")


Feature Engineering Complete
Final Shape: (180519, 52)

Remaining Missing Values:
0

Sample Features:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                         3                              4   
1  TRANSFER                         5                              4   
2      CASH                         4                              4   
3     DEBIT                         3                              4   
4   PAYMENT                         2                              4   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1        -249.089996          311.359985     Late delivery   
2        -247.779999          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk  Category Name Customer City Customer Country  ...  \
0              

# STEP 3: Encoding categorical variables

In [38]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder


# LOAD FEATURE-ENGINEERED DATASET

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(180519, 52)


In [39]:
# IDENTIFY CATEGORICAL COLUMNS


categorical_columns = df.select_dtypes(include=['object']).columns

print("\nCategorical Columns:")
print(categorical_columns)


Categorical Columns:
Index(['Type', 'Delivery Status', 'Customer City', 'Customer Country',
       'Customer State', 'Department Name', 'Order Status',
       'shipping date (DateOrders)', 'shipping_route'],
      dtype='object')


In [40]:
# LABEL ENCODING


# Store encoders if needed later
label_encoders = {}

for col in categorical_columns:

    # Initialize encoder
    le = LabelEncoder()

    # Convert to string to avoid issues
    df[col] = df[col].astype(str)

    # Fit and transform
    df[col] = le.fit_transform(df[col])

    # Save encoder
    label_encoders[col] = le

    print(f"Encoded: {col}")


# VERIFY RESULTS


print("\nEncoded Dataset Preview:")
print(df.head())

print("\nDataset Info:")
print(df.info())

Encoded: Type
Encoded: Delivery Status
Encoded: Customer City
Encoded: Customer Country
Encoded: Customer State
Encoded: Department Name
Encoded: Order Status
Encoded: shipping date (DateOrders)
Encoded: shipping_route

Encoded Dataset Preview:
   Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     1                         3                              4   
1     3                         5                              4   
2     0                         4                              4   
3     1                         3                              4   
4     2                         2                              4   

   Benefit per order  Sales per customer  Delivery Status  Late_delivery_risk  \
0          91.250000          314.640015                0                   0   
1        -249.089996          311.359985                1                   1   
2        -247.779999          309.720001                3                   0   
3          22.860001  

In [41]:
df['route_id'] = (
    df['shipping_route']
    .astype('category')
    .cat.codes
)

In [42]:
route_profit = (
    df.groupby('route_id')['Benefit per order']
      .mean()
      .to_dict()
)

df['avg_route_profit'] = df['route_id'].map(route_profit)

In [43]:
route_time = (
    df.groupby('route_id')['Days for shipping (real)']
      .mean()
      .to_dict()
)

df['avg_route_time'] = df['route_id'].map(route_time)

In [44]:
route_late = (
    df.groupby('route_id')['Late_delivery_risk']
      .mean()
      .to_dict()
)

df['route_late_rate'] = df['route_id'].map(route_late)

In [45]:
# SAVE ENCODED DATASET


output_file = 'dataco_rl_encoded.csv'

df.to_csv(output_file, index=False)

print(f"\nEncoded dataset saved as: {output_file}")


Encoded dataset saved as: dataco_rl_encoded.csv


# STEP 4: Train/test temporal split

In [46]:
df = pd.read_csv('dataco_rl_encoded.csv')

In [47]:
df = df.sort_values(
    by='shipping date (DateOrders)'
).reset_index(drop=True)


In [48]:
# TEMPORAL TRAIN/TEST SPLIT 80/20
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [49]:
print(f"Training Rows: {len(train_df)}")
print(f"Testing Rows : {len(test_df)}")

Training Rows: 144415
Testing Rows : 36104


In [50]:
print("\nTraining Date Range:")
print(
    train_df['shipping date (DateOrders)'].min(),
    "to",
    train_df['shipping date (DateOrders)'].max()
)

print("\nTesting Date Range:")
print(
    test_df['shipping date (DateOrders)'].min(),
    "to",
    test_df['shipping date (DateOrders)'].max()
)



Training Date Range:
0 to 51998

Testing Date Range:
51999 to 63700


In [51]:
train_df.to_csv(
    'dataco_rl_train.csv',
    index=False
)

test_df.to_csv(
    'dataco_rl_test.csv',
    index=False
)

print("\nTrain and Test datasets saved.")


Train and Test datasets saved.


In [52]:
  # CREATE RL STATE MATRICES
  state_columns = [
      'Market',
      'Order Region',
      'Category Name',
      'Order Item Quantity',
      'Sales',
      'Days for shipping (scheduled)',
      'shipping_mode_encoded'
  ]

  # Keep only columns that exist
  state_columns = [
      col for col in state_columns
      if col in df.columns
  ]

  # Training states
  X_train = train_df[state_columns].values

  # Testing states
  X_test = test_df[state_columns].values

  print("\nTraining State Shape:")
  print(X_train.shape)

  print("\nTesting State Shape:")
  print(X_test.shape)


Training State Shape:
(144415, 6)

Testing State Shape:
(36104, 6)


# STEP 4: Normalization/scaling ONLY FOR TRAINING DATA!!!!

In [53]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import joblib

In [54]:
train_df = pd.read_csv('dataco_rl_train.csv')
test_df = pd.read_csv('dataco_rl_test.csv')

print("Training Shape:", train_df.shape)
print("Testing Shape :", test_df.shape)

Training Shape: (144415, 56)
Testing Shape : (36104, 56)


In [55]:
# EXCLUDE TARGET / REWARD COLUMNS
numeric_columns = train_df.select_dtypes(
    include=['int64', 'float64']
).columns

print("\nNumeric Columns:")
print(numeric_columns.tolist())
exclude_columns = [
    'Order Profit Per Order'
]

scale_columns = [
    col for col in numeric_columns
    if col not in exclude_columns
]

print("\nColumns Being Scaled:")
print(scale_columns)


Numeric Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode', 'order_year', 'order_month', 'order_day', 'order_dayofweek', 'is_weekend', 'profit_margin', 'revenue_per_item', 'shipping_urgency', 'shipping_mode_encoded', 'shipping_route', 'total_order_value_per_quantity', 'large_order', 'high_profit_order', 'market_region', 'normaliz

In [56]:
scaler = MinMaxScaler()

# Fit ONLY on training data
scaler.fit(train_df[scale_columns])

MinMaxScaler()

In [57]:
train_df[scale_columns] = scaler.transform(
    train_df[scale_columns]
)

test_df[scale_columns] = scaler.transform(
    test_df[scale_columns]
)

In [58]:
print("\nScaled Training Dataset Preview:")
print(train_df.head())

print("\nScaled Testing Dataset Preview:")
print(test_df.head())


Scaled Training Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0  0.333333                  1.000000                            0.5   
1  0.333333                  1.000000                            0.5   
2  0.333333                  1.000000                            0.5   
3  0.666667                  0.833333                            0.5   
4  0.666667                  0.833333                            0.5   

   Benefit per order  Sales per customer  Delivery Status  Late_delivery_risk  \
0           0.837568            0.098577         0.333333                 1.0   
1           0.845509            0.172052         0.333333                 1.0   
2           0.829864            0.089252         0.333333                 1.0   
3           0.829736            0.029097         0.333333                 1.0   
4           0.831782            0.058680         0.333333                 1.0   

   Category Name  Customer City  Customer Coun

In [59]:
train_df.to_csv(
    'dataco_rl_train_scaled.csv',
    index=False
)

test_df.to_csv(
    'dataco_rl_test_scaled.csv',
    index=False
)

print("\nScaled datasets saved.")


Scaled datasets saved.


In [60]:
joblib.dump(
    scaler,
    'minmax_scaler.pkl'
)

print("Scaler saved as minmax_scaler.pkl")

Scaler saved as minmax_scaler.pkl


# STEP 5: Creating reward variables

In [61]:
# Profit Reward
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df['profit_reward'] = scaler.fit_transform(
    df[['Benefit per order']]
)

In [62]:
# Shipping Time Penalty
df['shipping_time_penalty'] = scaler.fit_transform(
    df[['Days for shipping (real)']]
)

df['shipping_time_reward'] = 1 - df['shipping_time_penalty']

In [63]:
# Late Delivery Penalty
df['on_time_reward'] = 1 - df['Late_delivery_risk']

In [64]:
df['reward'] = (
    0.5 * df['profit_reward']
    + 0.3 * df['on_time_reward']
    + 0.2 * df['shipping_time_reward']
)

# STEP 6: Building RL state-action structure

In [67]:
# Create Route IDs
df['shipping_route'] = (
    df['Order Region'].astype(str) + '_' +
    df['Customer Country'].astype(str)
)

df['route_id'] = (
    df['shipping_route']
    .astype('category')
    .cat.codes
)

In [68]:
# Create Order Value Buckets
df['order_value_bucket'] = pd.qcut(
    df['Sales'],
    q=5,
    labels=False
)

In [69]:
# Create Route Reliability Buckets
route_late = (
    df.groupby('route_id')['Late_delivery_risk']
      .mean()
)

df['route_late_rate'] = (
    df['route_id']
      .map(route_late)
)
df['route_late_bucket'] = pd.qcut(
    df['route_late_rate'],
    q=5,
    labels=False,
    duplicates='drop'
)

In [70]:
# Create Route Profit Buckets
route_profit = (
    df.groupby('route_id')['Benefit per order']
      .mean()
)

df['route_profit'] = (
    df['route_id']
      .map(route_profit)
)

df['route_profit_bucket'] = pd.qcut(
    df['route_profit'],
    q=5,
    labels=False,
    duplicates='drop'
)

In [72]:
    # Finalized State
state_columns = [
    'route_id',
    'Category Name',
    'order_value_bucket',
    'route_late_bucket',
    'route_profit_bucket'
]

#  State tuples
df['state'] = list(zip(
    df['route_id'],
    df['Category Name'],
    df['order_value_bucket'],
    df['route_late_bucket'],
    df['route_profit_bucket']
))

In [74]:
# Action Structure
# Select Shipping Mode
action_mapping = {
    0: 'Standard Class',
    1: 'Second Class',
    2: 'First Class',
    3: 'Same Day'
}
shipping_mode_map = {
    'Standard Class': 0,
    'Second Class': 1,
    'First Class': 2,
    'Same Day': 3
}

df['action'] = (
    df['Shipping Mode']
      .map(shipping_mode_map)
)
A = {0,1,2,3}

In [75]:
# Reward Structure
df['reward'] = (
    0.5 * df['profit_reward']
    + 0.3 * df['on_time_reward']
    + 0.2 * df['shipping_time_reward']
)

In [76]:
# Q-table Structure
from collections import defaultdict
import numpy as np

Q = defaultdict(
    lambda: np.zeros(4)
)


* State = information known BEFORE shipment decision
* Action = shipping method / route / priority decision
* Reward = profit − lateness penalty
* Next state = next order environment

Reward Function

Rt=α(Profit)−β(Late Shipment Penalty)

Where:

α controls profit importance

β controls delivery performance importance

A more detailed version:

Rt=αPt−βLt−γCt

Where:
Pt = profit

Lt = lateness indicator or delay days

Ct = shipping cost

# STEP 8: Q-learning or SARSA environment setup